# Train a Small Language Model from Scratch, Step by Step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/tinystories-ARM-model/blob/main/train_step_by_step.ipynb)

This notebook builds a small GPT-style language model (~30M parameters) from scratch and trains it on
[Mini FineWeb-Edu](https://huggingface.co/datasets/BaoMinhTran/mini-fineweb-edu), educational web pages
filtered from Common Crawl. [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories), simple
short stories, is one edit away in Step 1.

Run the cells **top to bottom**. Each step explains what it does and shows you something you can look at.

> This project is built for **Apple Silicon Macs**, where it trains on the Mac's GPU and comes with a
> step-by-step web app (`python3 app/server.py`). This notebook runs the training on a cloud GPU instead:
> **Google Colab** or **Kaggle**. Step 10 packages the trained model into one zip that you can bring back
> to your Mac with `python3 import_run.py <zip>`.

| Step | What happens |
|---|---|
| 0 | Setup: get the code, pick a device |
| 1 | Download the dataset (only once) |
| 2 | Tokenize: turn text into numbers |
| 3 | Make training batches |
| 4 | Build the model |
| 5 | Choose training settings |
| 6 | Train |
| 7 | Plot the loss curves |
| 8 | Generate text |
| 9 | Inference metrics on the GPU: the KV cache |
| 10 | Package the results to take home |

**On Colab:** *Runtime → Change runtime type → T4 GPU* (or L4 / A100 with Colab Pro).
**On Kaggle:** in the side panel, *Settings → Accelerator → GPU*, and switch **Internet on**.

## Step 0 · Setup

On Colab or Kaggle this clones the repository and installs the dependencies. Locally, open this notebook from
the repository folder (after `pip install -r requirements.txt`) and it uses the code that's already there.

In [ ]:
# @title Get the code
import os, sys

REPO_URL = "https://github.com/a-ramakrishnan/tinystories-ARM-model.git"
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PLATFORM = "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local"

if IN_COLAB or IN_KAGGLE:
    # Kaggle keeps everything under /kaggle/working as the notebook's output, so the code and data go to /tmp
    # and only the final zip is saved there.
    CLONE_DIR = "/content/tinystories-ARM-model" if IN_COLAB else "/tmp/tinystories-ARM-model"
    if not os.path.exists(CLONE_DIR):
        !git clone -q {REPO_URL} {CLONE_DIR}
    else:
        !git -C {CLONE_DIR} pull -q   # already cloned in this session: get the latest code
    %cd {CLONE_DIR}
    !pip install -q -r requirements.txt

assert os.path.exists("full-model.py"), "Open this notebook from the tinystories-ARM-model folder."
PROJECT_DIR = os.getcwd()
OUTPUT_DIR = "/kaggle/working" if IN_KAGGLE else PROJECT_DIR   # where the results zip is written
print(f"Running on {PLATFORM}. Project folder: {PROJECT_DIR}")

In [ ]:
# @title Import the project modules and pick a device
import torch

from data import BatchLoader, dataset_paths, get_dataset
from inference import generate_text, load_model, save_model_config
from model import GPT, GPTConfig, estimate_loss
from plots import plot_losses
from pretrain import pretrain
from tokenization import bin_files_exist, get_tokenizer, tokenize_and_save
from train_config import TrainConfig, setup_device

dev = setup_device()
print(f"Device: {dev.device}  |  precision: {dev.dtype}")
if dev.device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  |  memory: {props.total_memory / 1e9:.1f} GB  |  compute capability {props.major}.{props.minor}")
    !nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv
elif dev.device == "cpu":
    print("No GPU found: training will be slow. On Colab, switch to a GPU runtime.")

## Step 1 · Download the dataset

The default is Mini FineWeb-Edu's **small** subset: about 595,000 educational web pages, ~500M tokens
(1.3 GB download). Its other subsets are `debug` (~10M tokens), `tiny` (~100M), `base` (~1B) and `full`
(~2.9B). For short children's stories instead, set `dataset_name = "roneneldan/TinyStories"` and leave the
subset empty. You can switch to any text
dataset: a Hugging Face name (e.g. `Salesforce/wikitext` with subset `wikitext-2-raw-v1`), a Hugging Face
link, or a link to a `.txt`, `.jsonl`, `.csv` or `.parquet` file, such as
`https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt`.
Only text datasets are accepted. If there's no validation split, a small slice of train is held out.
Run `python3 full-model.py --list-datasets` for more datasets that work out of the box.

The first run downloads the dataset and saves a copy to `data/<dataset>/`; later runs load that copy instead
of downloading again. Each dataset keeps its own tokens and trained model in that folder.

In [ ]:
# @title Load the dataset
dataset_name = "BaoMinhTran/mini-fineweb-edu"  # @param {type:"string"}
dataset_config = "small"  # @param {type:"string"}
text_column = "text"  # @param {type:"string"}

paths = dataset_paths(dataset_name, dataset_config or None, os.path.join(PROJECT_DIR, "data"))
ds = get_dataset(paths.hf_dir, dataset_name, config_name=dataset_config or None, text_column=text_column)
print("Stored in:", os.path.relpath(paths.root, PROJECT_DIR))
print(ds)

In [ ]:
# @title Peek at a story
story_index = 0  # @param {type:"integer"}
print(ds["train"][story_index]["text"])

## Step 2 · Tokenize

Models work with numbers, not letters. The GPT-2 tokenizer splits text into **tokens** (whole words or
pieces of words) and gives each one an ID from a vocabulary of 50,257.

Every story is tokenized once and all the IDs are written into two files, `train.bin` and `validation.bin`.
If they already exist this step is skipped. *Tokenizing the full dataset takes a while the first time,
especially on Colab's small CPUs.*

In [ ]:
# @title Try the tokenizer
enc = get_tokenizer()
sample_text = "Photosynthesis turns sunlight into chemical energy."  # @param {type:"string"}
ids = enc.encode_ordinary(sample_text)
print("Token IDs:", ids)
print("Tokens:   ", [enc.decode([i]) for i in ids])
print(f"{len(sample_text)} characters -> {len(ids)} tokens")

In [ ]:
# @title Tokenize the whole dataset (skipped if already done)
import numpy as np

if bin_files_exist(paths.root):
    print("train.bin and validation.bin already exist, skipping.")
else:
    tokenize_and_save(ds, paths.root, num_proc=os.cpu_count())

for f in (paths.train_bin, paths.val_bin):
    n = len(np.memmap(f, dtype=np.uint16, mode="r"))
    print(f"{os.path.basename(f)}: {n:,} tokens")

## Step 3 · Make training batches

Training doesn't read stories one by one. Each **batch** is a set of random windows of `block_size`
consecutive tokens cut from `train.bin`. The **target** is the same window shifted one token to the right:
at every position, the model learns to predict the next token.

`block_size` is the model's *context window*: how many tokens it can look at once.

In [ ]:
# @title Batch settings
block_size = 256  # @param {type:"integer"}
batch_size = 32  # @param {type:"integer"}
seed = 42  # @param {type:"integer"}

torch.manual_seed(seed)
loader = BatchLoader(paths.root, block_size, batch_size, dev.device)
X, Y = loader.get_batch("train")
print("Inputs X:", tuple(X.shape), " Targets Y:", tuple(Y.shape), " (batch_size, block_size)")
print()
print("Input :", repr(enc.decode(X[0, :24].tolist())))
print("Target:", repr(enc.decode(Y[0, :24].tolist())))

## Step 4 · Build the model

A decoder-only transformer, like a miniature GPT-2:

- **Embeddings** turn each token ID and its position into a vector of size `n_embd`.
- **`n_layer` transformer blocks**, each with causal self-attention (`n_head` heads; every token looks only at
  earlier tokens) and a small feed-forward network.
- A final layer scores every word in the vocabulary as the possible next token.

Bigger numbers mean a more capable model, but slower training and more memory.

In [ ]:
# @title Model settings
# Presets: tiny 16M · small 30M (default) · medium 51M · large 124M (GPT-2 small size; fits easily on a T4)
model_size = "small"  # @param ["tiny", "small", "medium", "large", "custom"]
n_layer = 6  # @param {type:"integer"}
n_head = 6  # @param {type:"integer"}
n_embd = 384  # @param {type:"integer"}
dropout = 0.0  # @param {type:"number"}
pos_encoding = "learned"  # @param ["learned", "rope"]
norm = "layernorm"  # @param ["layernorm", "rmsnorm"]

PRESETS = {"tiny": (4, 4, 256), "small": (6, 6, 384), "medium": (8, 8, 512), "large": (12, 12, 768)}
if model_size != "custom":
    n_layer, n_head, n_embd = PRESETS[model_size]

config = GPTConfig(
    vocab_size=enc.n_vocab,
    block_size=block_size,
    n_layer=n_layer,
    n_head=n_head,
    n_embd=n_embd,
    dropout=dropout,
    bias=True,
    pos_encoding=pos_encoding,  # "rope": rotary positions (Llama-style), no position embeddings
    norm=norm,  # "rmsnorm": scale-only normalization (Llama-style)
)
torch.manual_seed(seed)
model = GPT(config).to(dev.device)
n_params = sum(p.numel() for p in model.parameters())
print(config)
print(f"Parameters: {n_params / 1e6:.1f}M")

In [ ]:
# @title Before training: loss and a sample
import math

with torch.no_grad():
    _, loss = model(X, Y)
print(f"Loss on one batch: {loss.item():.2f}")
print(f"Random guessing would score ln({enc.n_vocab}) = {math.log(enc.n_vocab):.2f}")
print()
model.eval()
print(generate_text(model, enc, "The water cycle is", dev.device, max_new_tokens=30))
model.train();

Gibberish, as expected: the weights are still random. Training fixes that.

## Step 5 · Training settings

- **quick**: 2,000 steps to see the whole pipeline work. The loss drops a lot, but the stories will still
  be rough.
- **full**: the settings `full-model.py` uses: 20,000 steps. On a T4 this takes one to a few hours,
  depending on the model size.

Both use a peak learning rate of `6e-4` decaying to `6e-5`, and update the weights after every batch
(`gradient_accumulation_steps = 1`). Earlier versions used `1e-4` and accumulated 8–32 batches per update,
which left the model with only a few hundred updates and badly undertrained. The learning rate warms up for
`warmup_steps`, then decays along a cosine curve down to `min_lr`.

**Resume.** Every evaluation saves the full training state (`training_state.pt`: the latest weights, the
optimizer's running averages, the step and the loss history) next to the checkpoint. Tick `resume` to carry
on from it: with the same number of steps, an interrupted run finishes as if it had never stopped; with more
steps (raise `full_steps`), a finished run is extended, with the learning-rate schedule stretched to the new
end. Files in `/content` are lost when a Colab session disconnects, so for long runs tick
`keep_checkpoints_in_drive` (Google Drive asks for permission once).


In [ ]:
# @title Choose a preset
mode = "quick"  # @param ["quick", "full"]
full_steps = 20000  # @param {type:"integer"}
learning_rate = 6e-4  # @param {type:"number"}
resume = False  # @param {type:"boolean"}
keep_checkpoints_in_drive = False  # @param {type:"boolean"}

best_model_path = paths.checkpoint
if keep_checkpoints_in_drive and IN_COLAB:
    # Checkpoints and the training state go to Google Drive, so they survive a disconnected session:
    # in a new session, run the cells again with resume ticked to carry on where the run stopped.
    from google.colab import drive
    drive.mount("/content/drive")
    run_dir = os.path.join("/content/drive/MyDrive/tinystories-runs", os.path.basename(paths.root))
    os.makedirs(run_dir, exist_ok=True)
    best_model_path = os.path.join(run_dir, "best_model_params.pt")
    print("Checkpoints and training state are saved in", run_dir)

if mode == "quick":
    train_cfg = TrainConfig(
        max_iters=2000, warmup_steps=200, eval_interval=250, eval_iters=50,
        learning_rate=learning_rate, min_lr=learning_rate / 10, batch_size=batch_size, seed=seed, best_model_path=best_model_path,
    )
else:
    train_cfg = TrainConfig(
        max_iters=full_steps, learning_rate=learning_rate, min_lr=learning_rate / 10, batch_size=batch_size, seed=seed,
        best_model_path=best_model_path,
    )
from pretrain import read_state_meta
saved_state = read_state_meta(os.path.dirname(best_model_path)) if resume else None
if resume and saved_state is None:
    print("Nothing to resume yet: this starts a new run.")
elif saved_state:
    print(f"Will resume at step {saved_state['step']:,} of {train_cfg.max_iters:,} "
          f"(best validation loss so far {saved_state['best_val']:.4f})")
train_cfg

## Step 6 · Train

Every `eval_interval` steps, the loss is measured on training and validation data. Whenever validation loss
reaches a new low, the weights are saved to `best_model_params.pt`.

Watch both numbers go down. If validation loss starts rising while training loss keeps falling, the model is
memorizing instead of learning (*overfitting*).

> To continue a stopped run, tick `resume` in Step 5 and run Step 5 and this cell again. To start over,
> re-run the *Model settings* cell in Step 4 first (and leave `resume` unticked).

In [ ]:
# @title Train the model
import time
from dataclasses import asdict

loader = BatchLoader(paths.root, config.block_size, train_cfg.batch_size, dev.device)
if dev.device == "cuda":
    torch.cuda.reset_peak_memory_stats()
start_step = saved_state["step"] if saved_state else 0
t0 = time.time()
train_losses, val_losses = pretrain(model, loader, train_cfg, dev, resume=bool(saved_state), model_config=asdict(config))
seconds = time.time() - t0
if os.path.abspath(train_cfg.best_model_path) != os.path.abspath(paths.checkpoint):
    import shutil
    shutil.copy(train_cfg.best_model_path, paths.checkpoint)  # the rest of the notebook reads it from here
best_val = min(val_losses) if val_losses else None
tokens = train_cfg.max_iters * train_cfg.batch_size * config.block_size
updates = -(-train_cfg.max_iters // train_cfg.gradient_accumulation_steps)
session_tokens = (train_cfg.max_iters - start_step) * train_cfg.batch_size * config.block_size
print(f"Trained {train_cfg.max_iters - start_step:,} steps" + (f" (resumed at step {start_step:,})" if start_step else "")
      + f" in {seconds / 60:.1f} min: {session_tokens / seconds:,.0f} tokens/s (including evaluations)")
print(f"{updates:,} optimizer updates × {n_params:,} parameters = {updates * n_params:,} parameter updates")
if dev.device == "cuda":
    print(f"Peak GPU memory while training: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB "
          f"of {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
gpu_name = torch.cuda.get_device_name(0) if dev.device == "cuda" else ("Apple GPU (MPS)" if dev.device == "mps" else "CPU")
train_summary = {
    "datasets": [dataset_name], "mix": None, "steps": train_cfg.max_iters, "updates": updates, "tokens": tokens,
    "params": n_params, "param_updates": n_params * updates, "best_val": best_val, "seconds": round(seconds, 1),
    "trained_at": time.strftime("%Y-%m-%d %H:%M"), "stopped_early": False, "device": dev.device, "chip": gpu_name,
    "platform": PLATFORM, "tokens_per_sec": round(session_tokens / seconds), "dtype": dev.dtype,
    "peak_gpu_memory": torch.cuda.max_memory_allocated() if dev.device == "cuda" else None,
}
save_model_config(paths.model_config, config, {**train_summary, "dataset_config": dataset_config or None,
                                               "train_config": asdict(train_cfg)})
print(f"Best validation loss: {best_val:.4f}" if best_val is not None else "No evaluation ran (max_iters < eval_interval).")

## Step 7 · Plot the loss curves

In [ ]:
# @title Training vs validation loss
RESULTS_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(RESULTS_DIR, exist_ok=True)
plot_losses(train_losses, val_losses, train_cfg.eval_interval, save_path=os.path.join(RESULTS_DIR, "loss.png"))

## Step 8 · Generate text

This loads the checkpoint with the best validation loss and continues each prompt (separate prompts with `|`).

- **temperature**: lower is safer and more repetitive, higher is more surprising (and more random).
- **top_k**: only pick among the `k` most likely next tokens. `0` means no limit.

Change the prompt and run the cell as many times as you like.

In [ ]:
# @title Generate text
prompts = "The water cycle is the process by which | In the history of science, | Once upon a time there was a little girl who"  # @param {type:"string"}
max_new_tokens = 200  # @param {type:"integer"}
temperature = 0.8  # @param {type:"slider", min:0.1, max:2.0, step:0.1}
top_k = 50  # @param {type:"integer"}

trained = load_model(config, train_cfg.best_model_path, dev.device)
torch.manual_seed(seed)
samples = []
for prompt in [p.strip() for p in prompts.split("|") if p.strip()]:  # one prompt per "|"
    text = generate_text(trained, enc, prompt, dev.device, max_new_tokens=max_new_tokens,
                         temperature=temperature, top_k=top_k or None)
    samples.append({"prompt": prompt, "text": text})
    print(text, "\n" + "-" * 80)
sample = ("\n\n" + "-" * 80 + "\n\n").join(s["text"] for s in samples)


## Step 9 · Inference on the GPU: the KV cache

Generating text is one forward pass per new token. Without a cache, each pass re-reads the whole context.
The **KV cache** keeps every layer's keys and values for the tokens already seen, so each new pass only
processes the new token. This cell generates the same text both ways (same random seed) and compares the
**time taken**, speed and memory on this GPU for a few lengths, then draws the comparison as a chart
(`kv_cache.png`, also in the Step 10 zip).

The cache costs `2 (key + value) × n_layer × n_embd × 4 bytes` per token. This model's positions are learned
per slot, so once the context window (`block_size`) is full the cache has to be rebuilt every step and the
advantage shrinks: watch the last row.

With a small model and a 256-token window, each step is dominated by the fixed cost of launching GPU work,
so expect a modest gain (on an Apple M5: about 1.1× for the Small model, 1.3× for Large). Try `model_size = "large"` in Step 4: the bigger
the model and the longer the context, the more the cache saves.

In [ ]:
# @title Benchmark generation with and without the KV cache
import matplotlib.pyplot as plt
from inference import benchmark_generation, device_memory, kv_bytes_per_token

prompt_ids = enc.encode_ordinary("The water cycle is the process by which")
per_token = kv_bytes_per_token(trained.config)
print(f"KV cache: {per_token / 1024:.1f} KB per token, {per_token * trained.config.block_size / 1e6:.2f} MB for a full "
      f"{trained.config.block_size}-token window  |  weights: {n_params * 4 / 1e6:.0f} MB")
print()
print(f"{'tokens':>7} | {'no cache':>9} | {'KV cache':>9} | {'no cache tok/s':>14} | {'KV cache tok/s':>14} | {'speedup':>7} | {'cache':>8} | {'rebuilds':>8} | same text")
bench_results = []
lengths = sorted({25, 60, 110, trained.config.block_size - len(prompt_ids), trained.config.block_size + 100})
for n in lengths:
    r = benchmark_generation(trained, prompt_ids, dev.device, n_tokens=n)
    bench_results.append({"n_tokens": n, **r})
    a, b = r["no_cache"], r["kv_cache"]
    print(f"{n:>7} | {a['total_ms'] / 1000:>8.2f}s | {b['total_ms'] / 1000:>8.2f}s | {a['tokens_per_sec']:>14} | {b['tokens_per_sec']:>14} | "
          f"{r['speedup']:>6}× | {b['kv_cache_bytes'] / 1e6:>6.2f}MB | {b['recomputes']:>8} | {r['same_output']}")
mem = device_memory(dev.device)
if mem.get("allocated") is not None:
    print(f"\nGPU memory in use: {mem['allocated'] / 1e6:.0f} MB" + (f", peak {mem['peak'] / 1e6:.0f} MB" if mem.get("peak") else ""))

# Chart: time taken for each length, with and without the cache
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
xs = range(len(bench_results))
ax1.bar([x - 0.2 for x in xs], [r["no_cache"]["total_ms"] / 1000 for r in bench_results], 0.4, label="no cache", color="#d9822b")
ax1.bar([x + 0.2 for x in xs], [r["kv_cache"]["total_ms"] / 1000 for r in bench_results], 0.4, label="KV cache", color="#2b7bd9")
ax1.set_xticks(list(xs), [str(r["n_tokens"]) for r in bench_results])
ax1.set_xlabel("new tokens generated"); ax1.set_ylabel("time taken (s)"); ax1.set_title("Time to generate"); ax1.legend()
ax2.plot([r["n_tokens"] for r in bench_results], [r["kv_cache"]["kv_cache_bytes"] / 1e6 for r in bench_results], "o-", color="#2b7bd9")
ax2.axvline(trained.config.block_size - len(prompt_ids), color="gray", ls="--", lw=1)
ax2.set_ylim(0, per_token * trained.config.block_size / 1e6 * 1.15)
ax2.text(trained.config.block_size - len(prompt_ids), per_token * trained.config.block_size / 1e6 * 1.08, "context window full ", color="gray", ha="right")
ax2.set_xlabel("new tokens generated"); ax2.set_ylabel("KV cache size (MB)"); ax2.set_title("KV cache memory")
fig.suptitle(f"KV cache on {torch.cuda.get_device_name(0) if dev.device == 'cuda' else dev.device}")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "kv_cache.png"), dpi=120)
plt.show()


## Step 10 · Package the results to take home

Everything from this run goes into one zip: the trained weights and their config, a summary of the run
(GPU, time, tokens per second, peak memory), the loss chart, the KV cache benchmark and its chart, and the generated samples.

- **Colab:** the zip downloads automatically (allow the download if your browser asks). It's also in the
  Files panel.
- **Kaggle:** the zip is saved to `/kaggle/working`; download it from the **Output** panel.

On your Mac, from the project folder, run `python3 import_run.py ~/Downloads/<zip name>`. It puts the model
in the right `data/` folder (backing up the one already there) and adds the run to the training history, so
the web app's Generate and *Inside the transformer* steps use your GPU-trained model.

Tick `include_training_state` to add the training state (about three times the size of the weights), so
the run can be continued later: `import_run.py` installs it, and `python3 full-model.py --resume` or the web
app's **Continue training** carries on from it.


In [ ]:
# @title Package the results
import json, time, zipfile
include_training_state = False  # @param {type:"boolean"}
from dataclasses import asdict
from inference import device_memory

slug = os.path.basename(paths.root)
stamp = time.strftime("%Y%m%d-%H%M")
zip_path = os.path.join(OUTPUT_DIR, f"tinystories-run-{slug}-{stamp}.zip")
run = {"format": 1, "slug": slug, "dataset": dataset_name, "dataset_config": dataset_config or None,
       "summary": {**train_summary, "model": {k: getattr(config, k) for k in ("n_layer", "n_head", "n_embd", "block_size")},
                   "max_iters": train_cfg.max_iters}}
metrics = {"gpu": device_memory(dev.device), "training": train_summary,
           "kv_cache_benchmark": globals().get("bench_results"), "config": asdict(config), "train_config": asdict(train_cfg)}
with zipfile.ZipFile(zip_path, "w") as z:
    z.write(paths.checkpoint, "best_model_params.pt")
    z.write(paths.model_config, "model_config.json")
    state_dir = os.path.dirname(train_cfg.best_model_path)
    if include_training_state and os.path.exists(os.path.join(state_dir, "training_state.pt")):
        for name in ("training_state.pt", "training_state.json"):  # so the run can be continued later
            z.write(os.path.join(state_dir, name), name)
    z.writestr("run.json", json.dumps(run, indent=2))
    z.writestr("metrics.json", json.dumps(metrics, indent=2, default=str))
    if os.path.exists(os.path.join(RESULTS_DIR, "loss.png")):
        z.write(os.path.join(RESULTS_DIR, "loss.png"), "loss.png")
    if os.path.exists(os.path.join(RESULTS_DIR, "kv_cache.png")):
        z.write(os.path.join(RESULTS_DIR, "kv_cache.png"), "kv_cache.png")
    if "sample" in globals():
        z.writestr("sample.txt", sample)
    if "samples" in globals():
        z.writestr("samples.json", json.dumps(samples, indent=2))
print(f"Saved {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
print(f"On your Mac:  python3 import_run.py ~/Downloads/{os.path.basename(zip_path)}")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## Next steps

- Try the **full** preset (raise `full_steps` for a longer run) for much better text, and a bigger model (`model_size = "large"`, 124M parameters) in Step 4.
- Compare the loss curves and the Step 9 speeds for different model sizes.
- Try another dataset in Step 1 and compare.
- Bring the model home with Step 10's zip and `python3 import_run.py`.
- To run everything without the notebook: `python3 full-model.py`.